In [142]:
from strands import Agent, tool
import logging

logging.getLogger("strands").setLevel(logging.ERROR)
logging.getLogger("strands.event_loop").setLevel(logging.ERROR)

In [143]:
LBS_TO_KG = 0.453592
INCH_TO_CM = 2.54


def lbs_to_kg(weight_lbs: float) -> float:
    """Convert weight from pounds to kilograms."""
    return weight_lbs * LBS_TO_KG


def feet_inches_to_cm(feet: int, inches: int) -> float:
    """Convert height from feet/inches to centimeters."""
    total_inches = (feet * 12) + inches
    return total_inches * INCH_TO_CM

In [144]:
import json
import re

def extract_json(text):
    """Extract a JSON object or array from an LLM response."""

    text = text.strip()

    # Remove Markdown code fences if present
    text = re.sub(r"^```(?:json)?", "", text)
    text = re.sub(r"```$", "", text)
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    match = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)

    if match:
        return json.loads(match.group(1))

    raise ValueError("No valid JSON found.")

In [145]:
def calculate_nutrition_targets(user_input):
    weight_lbs = user_input["weight_lbs"]
    age = user_input["age"]
    sex = user_input["sex"]
    activity = user_input["activity_level"]
    goal = user_input["goal"]

    weight_kg = lbs_to_kg(weight_lbs)
    height_cm = feet_inches_to_cm(
        user_input["height_feet"],
        user_input["height_inches"]
    )

    if sex.lower() == "male":
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161

    activity_levels = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725,
        "very active": 1.9,
    }

    tdee = bmr * activity_levels[activity]

    if goal.lower() == "fat loss":
        calorie_goal = tdee * 0.80
    elif goal.lower() == "muscle gain":
        calorie_goal = tdee * 1.10
    else:
        calorie_goal = tdee

    protein_goal = weight_lbs

    return {
        "bmr": round(bmr),
        "tdee": round(tdee),
        "calorie_goal": round(calorie_goal),
        "protein_goal": round(protein_goal),
    }

In [146]:
# Recipe sources allowed for nutrition searches
TRUSTED_DOMAINS = {
    "allrecipes.com",
    "eatingwell.com",
    "skinnytaste.com",
    "foodnetwork.com",
    "delish.com",
    "bbcgoodfood.com",
}

In [147]:
def is_recipe_page(url: str, title: str) -> bool:
    """
    Filter out collection pages and keep individual recipes.
    """

    blocked_terms = [
        "ideas",
        "best",
        "top",
        "collection",
        "roundup",
        "list",
        "high-protein",
        "meal-plan"
    ]

    url_lower = url.lower()
    title_lower = title.lower()

    # reject obvious collection pages
    for term in blocked_terms:
        if term in url_lower or term in title_lower:
            return False

    return True

In [148]:
from typing import List, Dict, Any
import requests
from bs4 import BeautifulSoup

def search_recipes_web(query: str, max_results: int = 10) -> List[Dict[str, Any]]:
    """Search trusted recipe websites for recipes matching a query."""

    url = "https://html.duckduckgo.com/html/"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        response = requests.post(
            url,
            data={"q": query},
            headers=headers,
            timeout=10
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        results = []
        
        for item in soup.find_all("div", class_="result")[:max_results]:
            link = item.find("a", class_="result__a")

            if not link:
                continue

            recipe_url = link.get("href", "")
            title = link.get_text(strip=True)

            snippet = item.find("a", class_="result__snippet")
            snippet_text = snippet.get_text(strip=True) if snippet else ""
            if any(domain in recipe_url.lower() for domain in TRUSTED_DOMAINS):
                results.append({
                    "title": title,
                    "url": recipe_url,
                    "snippet": snippet_text
                })

        return results

    except requests.RequestException:
        return []

In [149]:
def collect_recipe_results(calorie_target, ingredients):

    searches = [
        f"{ingredients} breakfast recipe",
        f"chicken rice broccoli recipe",
        f"high protein chicken recipe",
        f"protein snack recipe"
    ]


    results = []

    for query in searches:
        results.extend(search_recipes_web(query))

    return results

In [150]:
@tool
def recipe_search_tool(
    calorie_target: int,
    ingredients: str
):

    """
    Searches the web for high-protein recipes.
    """

    results =[]
    
    results = collect_recipe_results(
        calorie_target=calorie_target,
        ingredients=ingredients
    )


    return results

In [151]:
# Agent 1: Recipe Finder Agent

recipe_agent = Agent(
    name="RecipeResearchAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    callback_handler=None,
    system_prompt="""
You are a recipe generation and retrieval specialist.

Your task is to create structured recipe candidates using:
- web search results
- user ingredients
- nutrition targets

Use web results as inspiration.

If nutrition information is unavailable:
- generate realistic nutrition estimates
- create recipes using available ingredients

Do not create a meal plan.
Only create individual recipe candidates.

Inputs:
- calorie target
- protein target
- available ingredients
- recipe search results

Select recipes that:
- are high protein (25-50g per serving)
- fit the calorie target
- use available ingredients when possible
- come from reliable sources

Do NOT:
- create a meal plan
- modify recipes
- Nutrition values must be realistic estimates based on standard serving sizes.

Exclude recipes if required nutrition data is missing.

If a result is a collection page:
- Extract individual recipes mentioned in the snippet only if a URL is available.
- Otherwise ignore it.

If no recipes contain complete nutrition information:
return []

Never generate recipes from ingredients.
Never estimate calories.

Do not include URLs.
Do not include source metadata.
Return only recipe information.

Return ONLY valid JSON.

Each recipe must follow:

{
"name": string,
"meal_type": "breakfast | lunch | dinner | snack",
"calories": number,
"protein_g": number,
"carbs_g": number,
"fat_g": number,
"ingredients": [],
}

Output:

[
  {
    "name": "",
    "meal_type": "",
    "calories": 0,
    "protein_g": 0,
    "carbs_g": 0,
    "fat_g": 0,
    "ingredients": [],
    "url": ""
  }
]
"""
)

In [152]:
@tool
def recipe_finder_tool(
    calorie_target: int,
    protein_target: int,
    ingredients: str,
):
    
    """
    Finds high-protein recipes and generates meal options
    based on user nutrition targets.
    """

    search_results = recipe_search_tool(
        calorie_target=calorie_target,
        ingredients=ingredients,
    )

    prompt = f"""
You are a recipe extraction agent.

Your job:
Create realistic recipe options that a meal optimization agent can use to build a daily meal plan.

User nutrition targets:
Calories: {calorie_target} kcal
Protein: {protein_target} g

Available ingredients:
{ingredients}

Search results:
{json.dumps(search_results, indent=2)}

Recipe Requirements:

Generate realistic high-protein meal options.

Return:
- At least 2 breakfast options
- At least 3 lunch options
- At least 3 dinner options
- At least 2 snack options

Nutrition Guidelines:
- Estimate realistic calories and macros.
- Avoid extreme protein amounts.
- Do not create meals with unrealistic serving sizes.
- Include balanced combinations of protein, carbohydrates, and fats.
- Meals should be practical for everyday eating.

Ingredient Guidelines:
- Prioritize the provided ingredients.
- Use common supporting ingredients when needed.
- Do not replace the user's ingredients unnecessarily.
- Do not create recipes unrelated to the available ingredients.

Search Result Guidelines:
- Use search results only as inspiration.
- Do not copy recipes blindly.
- Extract useful meal ideas.

Output Rules:
- Return ONLY valid JSON.
- Do not include explanations.
- Do not include markdown.
- Do not include commentary.

Format:

[
    {{
        "name": "",
        "meal_type": "",
        "calories": 0,
        "protein_g": 0,
        "carbs_g": 0,
        "fat_g": 0,
        "ingredients": []
    }}
]
"""

    response = recipe_agent(prompt)

    response_text = response.message["content"][0]["text"]

    return extract_json(response_text)

In [153]:
@tool
def validate_meal_plan(
    meals: list,
    calorie_target: int,
    protein_target: int
):

    total_calories = sum(
        meal["calories"]
        for meal in meals
    )

    total_protein = sum(
        meal["protein_g"]
        for meal in meals
    )


    calories_valid = (
        abs(total_calories - calorie_target)
        <= calorie_target * 0.10
    )


    protein_valid = (
        total_protein >= protein_target
    )


    meal_types = [
        meal["meal_type"].lower()
        for meal in meals
    ]


    structure_valid = (
        meal_types.count("breakfast") == 1
        and meal_types.count("lunch") == 1
        and meal_types.count("dinner") == 1
        and meal_types.count("snack") <= 1
    )

    macro_calories_valid = True

    for meal in meals:
        calculated = (
            meal["protein_g"] * 4 +
            meal["carbs_g"] * 4 +
            meal["fat_g"] * 9
        )

        if abs(calculated - meal["calories"]) > 75:
            macro_calories_valid = False


    is_valid = (
        calories_valid
        and protein_valid
        and structure_valid
        and macro_calories_valid
    )


    feedback = []


    if not calories_valid:
        difference = total_calories - calorie_target

        feedback.append(
            f"Calories are {abs(difference)} kcal "
            f"{'over' if difference > 0 else 'under'} target."
        )


    if not protein_valid:
        difference = total_protein - protein_target

        feedback.append(
            f"Protein is {abs(difference)}g below target."
        )


    if not structure_valid:
        feedback.append(
            "Meal structure invalid. Need exactly one breakfast, lunch, dinner, and optional snack."
        )


    return {
        "is_valid": is_valid,
        "feedback": " ".join(feedback),
        "daily_totals": {
            "calories": total_calories,
            "protein_g": total_protein
        },
        "meals": meals
    }

In [154]:
# Agent 2: Meal Plan Optimizer Agent

optimizer_agent = Agent(
    name="MealPlanningAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="""
You are a meal planning optimizer.

Your task is to create a daily meal plan using provided recipes.

Inputs:
- User calorie target
- User protein target
- Available recipes

Goals:
1. Select breakfast, lunch, and dinner.
2. Include at most ONE snack. Only add a snack if breakfast, lunch, and dinner cannot reach calorie targets.
3. Match calorie target within 10%.
4. Match protein target within 90%-110%.
5. Prioritize balanced meals over maximum protein.

Meal rules:
- Exactly one breakfast.
- Exactly one lunch.
- Exactly one dinner.
- Do not duplicate meal types.
- Prefer recipes using user ingredients.

If calories are too low:
Increase portions or add realistic sides:
- rice
- oats
- fruit
- peanut butter
- avocado
- olive oil

Do not fix calorie shortages by adding unnecessary protein-heavy foods.

Before returning:
1. Calculate total calories and protein.
2. The orchestrator will validate the result separately.
3. Do not call validation tools.
4. Return only the final valid JSON.

Output format:

{
  "meals": [
    {
      "name": "",
      "meal_type": "",
      "calories": 0,
      "protein_g": 0,
      "carbs_g": 0,
      "fat_g": 0,
      "ingredients": []
    }
  ],
  "daily_total": {
    "calories": 0,
    "protein_g": 0
  }
}

Rules:
- Output ONLY JSON.
- No markdown.
- No explanations.
""",
    tools=[validate_meal_plan],
    callback_handler=None,
)

In [155]:
@tool
def meal_optimizer_tool(recipes, calorie_target, protein_target, feedback=""):
    
    """
    Generate a meal plan from available recipes.
    """

    feedback_section = ""

    if feedback:
        feedback_section = f"""
Previous validation feedback:
{feedback}

Revise the meal plan to satisfy this feedback.
"""

    optimizer_prompt = f"""
You are a nutrition optimization agent.

Your task:
Generate a single validated daily meal plan that satisfies the user's nutrition targets.

User nutrition targets:
Calories: {calorie_target} kcal
Protein: {protein_target} g

Available recipes:
{json.dumps(recipes, indent=2)}

{feedback_section}

CRITICAL OUTPUT RULE:

Your response is parsed automatically by Python.

You MUST NOT output:
- explanations
- calculations
- analysis
- thoughts
- progress updates
- selection reasoning
- validation comments

Your entire response MUST be a single JSON object.

MEAL STRUCTURE RULES:
- Select exactly ONE breakfast.
- Select exactly ONE lunch.
- Select exactly ONE dinner.
- Select ZERO or ONE snack only if required to reach nutrition targets.
- Never create duplicate meal types.
- Never remove required meals during revisions.
- Meals should represent realistic daily eating patterns.
- If the plan is within 10% of calories and within 15% of protein, return the plan without modifications. Do not optimize further.

Protein optimization:
- Treat protein as a minimum requirement, not the primary optimization goal.
- Once protein reaches the target, prioritize calories using carbohydrates and fats.
- Avoid adding additional chicken, eggs, or protein-heavy ingredients after protein target is met.
- Prefer calorie increases from:
  rice, oats, pasta, potatoes, avocado, olive oil, peanut butter, nuts, dairy.


Recipe modification rules:
- Do not modify recipe nutrition values unless portion changes are explicitly stated.
- If increasing calories, update ingredients to reflect the change.
- Add modifications as separate ingredients.
- Recalculate macros after modifications.
- Never silently change calories/macros.

NUTRITION RULES:
- Final calories MUST be within +/-10% of the calorie target.
- Protein should be between 100%-115% of target.
- Avoid exceeding 120% of target.
- Avoid unnecessary protein overages.
- Prioritize calorie accuracy over maximizing protein.

CALORIE ADJUSTMENT RULES:
Before returning:
1. Calculate the total calories from all meals.
2. Compare against the calorie target.
3. If calories are too low:
   - Increase portions first.
   - Add calorie-dense foods such as:
       - extra rice
       - oats
       - olive oil
       - avocado
       - peanut butter
       - nuts
       - dairy
4. If calories are too high:
   - Reduce portions or remove calorie additions.
5. Only increase protein sources if protein is below target.

REVISION RULES:
If previous validation feedback is provided:
- Fix ONLY the issues mentioned in the feedback.
- Preserve the meal structure.
- Do not replace the entire meal plan unless necessary.
- Do not ignore validation feedback.

INGREDIENT RULES:
- Use the provided recipes as the source of meals.
- Do not recreate or rewrite the recipe database.
- Do not ask for a different input format.
- Do not invent unrealistic nutrition values.
- Do not add unrelated foods unless they are calorie adjustments.

OUTPUT RULES:
Do not show your calculations.
Do not describe your selection process.
Do not explain decisions.
Do not output reasoning.

Your entire response must be ONLY the JSON meal plan.

Schema:

{{
 "meals": [
   {{
    "name": "",
    "meal_type": "",
    "calories": 0,
    "protein_g": 0,
    "carbs_g": 0,
    "fat_g": 0,
    "ingredients": []
   }}
 ],
 "daily_total": {{
    "calories": 0,
    "protein_g": 0
 }}
}}
"""


    meal_plan_result = optimizer_agent(optimizer_prompt)

    meal_plan_text = meal_plan_result.message["content"][0]["text"]

    return extract_json(meal_plan_text)

In [156]:
def print_header():
    print("""
==================================================
                  CUTBUDDY AI 
        MULTI-AGENT NUTRITION ASSISTANT
==================================================

    Powered by Strands Multi-Agent Workflow
    """)
def print_nutrition_summary(targets):
    print("""
==================================================
                NUTRITION PROFILE
==================================================
""")

    print(f"""
BMR: {targets['bmr']} kcal/day
TDEE: {targets['tdee']} kcal/day

Daily Target:
Calories: {targets['calorie_goal']} kcal
Protein: {targets['protein_goal']}g
""")

In [ ]:
def print_meal_plan(meal_plan):

    print("""
==================================================
                 DAILY MEAL PLAN
==================================================
""")

    for meal in meal_plan.get("meals", []):

        print(f"""
{meal.get("meal_type", "MEAL").upper()}
--------------------------------------------------
{meal.get("name", "Unnamed Meal")}

Calories: {meal.get("calories", 0)} kcal
Protein: {meal.get("protein_g", 0)} g
Carbs: {meal.get("carbs_g", 0)} g
Fat: {meal.get("fat_g", 0)} g

Ingredients:
""")

        for ingredient in meal.get("ingredients", []):
            print(f"  • {ingredient}")

    totals = meal_plan.get("daily_total", {})

    print("""
==================================================
                  DAILY TOTALS
==================================================
""")

    print(f"""
Calories: {totals.get("calories", 0)} kcal
Protein: {totals.get("protein_g", 0)} g
""")

    print("""
==================================================
""")
    print("✓ Meal plan created successfully")

In [158]:
orchestrator_agent = Agent(
    name="MealPlanningOrchestrator",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[
        recipe_finder_tool,
        meal_optimizer_tool,
        validate_meal_plan,
    ],
    callback_handler=None,
    system_prompt="""
You are a workflow orchestration agent for a nutrition planning system.

Your only responsibility is coordinating tools and returning the final validated meal plan.

Available tools:

1. recipe_finder_tool
- Searches and extracts recipe options based on user ingredients and nutrition targets.

2. meal_optimizer_tool
- Creates or revises a daily meal plan using available recipes.

3. validate_meal_plan
- Checks calories, protein, and meal structure constraints.

WORKFLOW:

Step 1:
Call recipe_finder_tool exactly once.

Step 2:
Send the recipe results to meal_optimizer_tool to generate a meal plan.

Step 3:
Call validate_meal_plan on the generated meal plan.

VALIDATION LOOP:

A valid meal plan requires:
- Calories within +/-10% of calorie target.
- Protein within acceptable range.
- Correct meal structure:
    - exactly one breakfast
    - exactly one lunch
    - exactly one dinner
    - zero or one snack

If validation returns is_valid=true:
- Do not call any more tools.
- Extract the "meals" field from the validation result.
- Return ONLY the validated meal plan JSON.

If validation returns is_valid=false:
- Send the validation feedback to meal_optimizer_tool.
- Generate a revised meal plan.
- Validate again.

Maximum attempts:
- Maximum 3 meal optimization attempts.
- Maximum 3 validation calls.

Never return an invalid meal plan.

TOOL USAGE RULES:

- Never call recipe_finder_tool more than once.
- Never call validation after successful validation.
- Never call tools after a valid meal plan is found.
- Do not manually create meal plans.
- Do not modify tool outputs yourself.

OUTPUT RULES:

You are not a conversational assistant.

Never output:
- reasoning
- explanations
- progress updates
- retry messages
- tool descriptions
- apologies
- comments about what you are doing

Return ONLY the final JSON object.

The final JSON must exactly follow:

{
  "meals": [
    {
      "name": "",
      "meal_type": "",
      "calories": 0,
      "protein_g": 0,
      "carbs_g": 0,
      "fat_g": 0,
      "ingredients": []
    }
  ],
  "daily_total": {
    "calories": 0,
    "protein_g": 0,
    "carbs_g": 0,
    "fat_g": 0
  } 
}
"""
)

In [159]:
def main():

    user_input = {
        "weight_lbs": 153,
        "height_feet": 5,
        "height_inches": 7,
        "age": 20,
        "sex": "female",
        "activity_level": "moderate",
        "ingredients": "chicken, eggs, rice, broccoli",
        "goal": "fat loss",
    }

    print_header()

    targets = calculate_nutrition_targets(user_input)

    print_nutrition_summary(targets)

    prompt = f"""
Generate a personalized daily meal plan.

Nutrition Targets:
- Calories: {round(targets["calorie_goal"])}
- Protein: {round(targets["protein_goal"])}g

Available Ingredients:
{user_input["ingredients"]}

Goal:
{user_input["goal"]}

Coordinate the available tools to produce a validated meal plan.
Return only the final meal plan.
"""
    response = orchestrator_agent(prompt)

    final_plan = response.message["content"][0]["text"]

    meal_plan = extract_json(final_plan)

    print_meal_plan(meal_plan)

In [160]:
main()


                  CUTBUDDY AI 
        MULTI-AGENT NUTRITION ASSISTANT

    Powered by Strands Multi-Agent Workflow
    

                NUTRITION PROFILE


BMR: 1497 kcal/day
TDEE: 2320 kcal/day

Daily Target:
Calories: 1856 kcal
Protein: 153g


                DAILY MEAL PLAN


BREAKFAST
--------------------------------------------------
Chicken and Egg Breakfast Bowl

Calories: 420 kcal
Protein: 42 g
Carbs: 24 g
Fat: 16 g

Ingredients:

  • 4 oz grilled chicken
  • 2 eggs
  • 0.5 cup cooked rice
  • 0.5 cup broccoli
  • oil

LUNCH
--------------------------------------------------
Grilled Chicken with Rice and Broccoli

Calories: 520 kcal
Protein: 45 g
Carbs: 48 g
Fat: 12 g

Ingredients:

  • 6 oz grilled chicken breast
  • 1 cup cooked rice
  • 1.5 cups broccoli
  • olive oil
  • garlic
  • salt
  • pepper

DINNER
--------------------------------------------------
Baked Chicken, Rice, and Broccoli Casserole

Calories: 580 kcal
Protein: 48 g
Carbs: 56 g
Fat: 14 g

Ingredients:

  